## 기계학습프로그래밍 실습 2주차 전처리

In [ ]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

print(df.shape)

#891개의 행과 15개의 열이 있음

(891, 15)


In [ ]:
print(df[["survived", "pclass", "sex", "age", "fare", "embarked"]].head())
print(df.dtypes.head(8))

   survived  pclass     sex   age     fare embarked
0         0       3    male  22.0   7.2500        S
1         1       1  female  38.0  71.2833        C
2         1       3  female  26.0   7.9250        S
3         1       1  female  35.0  53.1000        S
4         0       3    male  35.0   8.0500        S
survived      int64
pclass        int64
sex          object
age         float64
sibsp         int64
parch         int64
fare        float64
embarked     object
dtype: object


In [ ]:
na = df.isna().sum()
print(na[na > 0 ])

df = df.drop(columns=["deck"])
print("남은 열 개수:", df.shape[1])

age            177
embarked         2
deck           688
embark_town      2
dtype: int64
남은 열 개수: 14


- 5% 미만: 어떤 방법을 써도 결과가 비슷함. 간단한 방법을 적용
- 5% ~ 20% : 데이터를 채움. 결측 표시 열을 함께 만드는 것을 권장
- 20% ~ 50% : 채우긴 채우되 신중하게 채움. 그 열이 정말 필요한 지 검토
- 50% 초과 : 버리는 것을 우선 검토

In [ ]:
print(df[["age", "fare"]].describe().round(2))

          age    fare
count  714.00  891.00
mean    29.70   32.20
std     14.53   49.69
min      0.42    0.00
25%     20.12    7.91
50%     28.00   14.45
75%     38.00   31.00
max     80.00  512.33


- count : 결측이 아닌 개수 / 열마다 다르면 결측이 있다는 뜻
- mean : 평균 / 중앙값과 크게 다르면 치우친 분포
- std : 표준편차 / 평균보다 크면 퍼짐이 심함.
- 25% : 1사분위수 / 이상치 경계 계산에 쓰임
- 50% : 중앙값 / 평균과 비교
- 75% : 3사분위수 / 이상치 경계 계산에 쓰임
- max : 최댓값 / 평균의 몇 배인지 확인

### 훈련과 테스트 분할

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["survived"])
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("훈력:", X_train.shape, "/ 테스트:", X_test.shape)
print("전체 생존율 :", round(y.mean(), 4))
print("훈련 생존율 :", round(y_train.mean(), 4))
print("테스트 생존율:", round(y_test.mean(), 4))

훈력: (712, 13) / 테스트: (179, 13)
전체 생존율 : 0.3838
훈련 생존율 : 0.3834
테스트 생존율: 0.3855


In [ ]:
_, _, y_tr_bad, y_te_bad = train_test_split(X, y, test_size=0.2, random_state=39)
print("stratify 없이 -> 훈련", round(y_tr_bad.mean(), 4),"/ 테스트", round(y_te_bad.mean(), 4))

_, _, y_tr_ok, y_te_ok = train_test_split(X, y, test_size=0.2, random_state=39, stratify=y)
print("stratify 주면 -> 훈련", round(y_tr_ok.mean(), 4),
"/ 테스트", round(y_te_ok.mean(), 4))


stratify 없이 -> 훈련 0.4017 / 테스트 0.3128
stratify 주면 -> 훈련 0.3834 / 테스트 0.3855


### 결측치

In [ ]:
ratio = df.isna().mean() * 100
print(ratio[ratio > 0].round(2))

df.info()

age            19.87
embarked        0.22
embark_town     0.22
dtype: float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  embark_town  889 non-null    object 
 12  alive        891 non-null    object 
 13  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(6)
memory usage: 85.4+ KB


In [ ]:
#빈칸채우기
from sklearn.impute import SimpleImputer

X_train = X_train.copy()
X_test = X_test.copy()

imputer = SimpleImputer(strategy="median")
imputer.fit(X_train[["age"]]) # 훈련에서만 중앙값을 구합니다.

print("전체에서 구한 중앙값:", df["age"].median())
print("훈련에서 구한 중앙값:", imputer.statistics_[0])

전체에서 구한 중앙값: 28.0
훈련에서 구한 중앙값: 28.5


In [ ]:
#빈칸채우기 (이어서)
print("채우기 전 평균/표준편차:",
      round(X_train["age"].mean(), 2), round(X_train["age"].std(), 2))

X_train["age"] = imputer.transform(X_train[["age"]])
X_test["age"] = imputer.transform(X_test[["age"]]) #같은 값을 적용

print("채운 후 평균/표준편차",
      round(X_train["age"].mean(), 2), round(X_train["age"].std(), 2))

채우기 전 평균/표준편차: 29.81 14.49
채운 후 평균/표준편차 29.56 13.03


### 이상치

In [ ]:
#이상치 찾기
q1 = X_train["fare"].quantile(0.25)
q3 = X_train["fare"].quantile(0.75)
iqr = q3 - q1
high = q3 + 1.5 * iqr

print("Q1:", round(q1, 2), "/ Q3:", round(q3, 2), "/ IQR: ", round(iqr, 2))
print("정상 범위 상한:", round(high, 2))
print("상한을 넘는 건수:", int((X_train["fare"] > high).sum()), "/", len(X_train))



Q1: 7.9 / Q3: 31.0 / IQR:  23.1
정상 범위 상한: 65.66
상한을 넘는 건수: 91 / 712


In [ ]:
#값 확인
print("가장 비싼 표 세건:",
      X_train["fare"].sort_values(ascending=False).head(3).round(2).tolist())

# 512.33을 낸 승객이 누구인지 봅니다
expensive = df[df["fare"] > 500][["pclass", "sex", "age", "survived"]]
print(expensive)

가장 비싼 표 세건: [512.33, 512.33, 263.0]
     pclass     sex   age  survived
258       1  female  35.0         1
679       1    male  36.0         1
737       1    male  35.0         1


In [ ]:
#원핫 인코딩(수동버전)
cols = ["sex", "embarked"]
X_train_enc = pd.get_dummies(X_train[cols])
X_test_enc = pd.get_dummies(X_test[cols])

print("만들어진 열:", list(X_train_enc.columns))
print(X_train_enc.head(3))

만들어진 열: ['sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S']
     sex_female  sex_male  embarked_C  embarked_Q  embarked_S
692       False      True       False       False        True
481       False      True       False       False        True
527       False      True       False       False        True


In [ ]:
# 훈련에는 있고 테스트에는 없는 값이 있으면 열 개수가 달라집니다
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print("훈련 열 개수:", X_train_enc.shape[1],
      "/ 테스트 열 개수:", X_test_enc.shape[1])

훈련 열 개수: 5 / 테스트 열 개수: 5


[스케일 방법들의 수식과 성질]
- 최소 최대 정규화 : 이상치가 있는 경우 취약함 x` = (x-최솟값)/(최댓값-최솟값)
- 표준화 : 가장 널리 쓰임 x` = (x-평균)/표준편차
- 로비스트 조정 : 이상치가 많을때 씀 x` = (x-중앙값)/IQR

In [ ]:
#파이프라인으로 묶기
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

num_cols = ["age", "fare", "sibsp", "parch"]

pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

pipe.fit(X_train[num_cols])     #훈련에서만 fit
X_train_ready = pipe.transform(X_train[num_cols])
X_test_ready = pipe.transform(X_test[num_cols])

In [ ]:
print("훈련 평균 : ", X_train_ready.mean(axis=0).round(3))
print("훈련 표준편차:", X_train_ready.std(axis=0).round(3))
print("테스트 평균 : ", X_test_ready.mean(axis=0).round(3))
print("테스트 표준편차:", X_test_ready.std(axis=0).round(3))

훈련 평균 :  [ 0. -0. -0. -0.]
훈련 표준편차: [1. 1. 1. 1.]
테스트 평균 :  [-0.036  0.04   0.141 -0.053]
테스트 표준편차: [0.995 1.16  1.178 0.791]


In [ ]:
print("전처리 전 age 범위:", X_train ["age"].min(), "에서", X_train["age"].max())
print("전처리 전 fare 범위:", round(X_train ["fare"].min(), 2), "에서",
      round(X_train["fare"].max(), 2))
print("전처리 후 age 범위:", round(X_train_ready[:, 0].min(), 2), "에서",
      round(X_train_ready[:, 0].max(), 2))
print("전처리 후 fare 범위:", round(X_train_ready[:, 1].min(), 2), "에서",
      round(X_train_ready[:, 1].max(), 2))

전처리 전 age 범위: 0.42 에서 80.0
전처리 전 fare 범위: 0.0 에서 512.33
전처리 후 age 범위: -2.24 에서 3.88
전처리 후 fare 범위: -0.66 에서 10.01
